In [ ]:
import mysql.connector
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Hoangtrieu1409",
    database="mavenfuzzyfactory"
)

cursor = conn.cursor()

query1 = """
SELECT YEAR(website_sessions.created_at) AS `YEAR`,
		YEARWEEK(website_sessions.created_at) AS `WEEK`,
        COUNT(website_sessions.website_session_id) AS sessions,
        COUNT(orders.order_id) AS orders,
         COUNT(orders.order_id)/COUNT(website_sessions.website_session_id) AS cvr FROM website_sessions
        LEFT JOIN orders ON website_sessions.website_session_id = orders.website_session_id
        GROUP BY 1,2
        ORDER BY 1,2 ASC;
"""
df1 = pd.read_sql(query1,conn)

X = df1['sessions']
Y = df1['cvr']

# regression
z = np.polyfit(X, Y, 1)
p = np.poly1d(z)

Y_pred = p(X)

# R2
ss_res = np.sum((Y - Y_pred)**2)
ss_tot = np.sum((Y - np.mean(Y))**2)
r2 = 1 - ss_res/ss_tot

# Adjusted R2
n = len(X)
k = 1
r2_adj = 1 - (1-r2)*(n-1)/(n-k-1)

print("R² =", round(r2,4))
print("Adjusted R² =", round(r2_adj,4))
plt.figure(figsize=(8,6))

# scatter plot
plt.scatter(df1['sessions'], df1['cvr'])

# tạo trend line
z = np.polyfit(df1['sessions'], df1['cvr'], 1)   # hồi quy tuyến tính
p = np.poly1d(z)

plt.plot(df1['sessions'], p(df1['sessions']))

plt.xlabel('Sessions')
plt.ylabel('Conversion Rate')
plt.title('Traffic vs Conversion Rate')

plt.grid(True, alpha=0.3)

plt.show()